# Practice 3 · Training Loop in Code

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

The optimizer from Topic 6 only *knows how* to move the weights; something still has to zero the
old gradients, run the forward pass, compute the loss, back-propagate and step — once per
mini-batch, once per epoch.

### II. Task Requirements

1. **Q1** — write *train_one_epoch(model, loader, criterion, optimizer, device)*.
2. **Q2** — write *fit(model, train_loader, test_loader, optimizer, epochs, device, scheduler=None)*.
3. Run the last cell: SGD and Adam on a 3000-image subset, two epochs each.

Steps: open the notebook in Haima, write the 🟢 parts, run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *train_one_epoch(...)* | mean training loss of one epoch | a *float*, about 2.0–2.4 in epoch 1 |
| *fit(...)* | training history | a dict with *train_loss* and *test_acc* lists |
| *test_acc* | accuracy on the test loader | between 0 and 1, about 0.30–0.40 after 2 epochs |
| *train_loss* | one entry per epoch | drops from epoch 1 to epoch 2 |

### IV. Who does what

- **Q1**: required for undergraduate students (postgraduates do it as well).
- **Q2**: required for postgraduate students; optional for undergraduates.
  An undergraduate who also finishes Q2 **gets bonus marks**; leaving it unfinished
  **costs nothing**.


## Question 1 · One epoch

Write *train_one_epoch(model, loader, criterion, optimizer, device)* that runs the five steps for
every mini-batch — *zero_grad* → *forward* → *loss* → *backward* → *step* — and returns the
**mean training loss over the epoch** as a Python *float*.

The model must be left in training mode, and the loss must be averaged over all samples (not over
batches), so that a short epoch and a long epoch are comparable.


In [1]:
# 🟢 EDIT: fill in the body of train_one_epoch() and run the cell.
# 🟡 READ: the data, the loaders and make_model() are given above.
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 🟡 READ: CIFAR10 ships with the assignment; ./data holds cifar-10-batches-py
DATA_ROOT = "data"
if not os.path.exists(os.path.join(DATA_ROOT, "cifar-10-batches-py")):
    raise FileNotFoundError(
        "CIFAR-10 not found under ./data - please contact the teaching assistant "
        "and copy the folder cifar-10-batches-py next to this notebook."
    )

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2613)
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

N_SUB, BATCH = 3000, 64
full_train = CIFAR10(DATA_ROOT, train=True, download=False, transform=tf)
full_test = CIFAR10(DATA_ROOT, train=False, download=False, transform=tf)
train_loader = DataLoader(Subset(full_train, range(N_SUB)), batch_size=BATCH, shuffle=True)
test_loader = DataLoader(full_test, batch_size=256)


# 🟡 READ: the network used by both questions
def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(3 * 32 * 32, 256), nn.ReLU(),
        nn.Linear(256, 10),
    )


print("train batches per epoch:", len(train_loader), "| test batches:", len(test_loader))


def train_one_epoch(model, loader, criterion, optimizer, device="cpu"):
    """Run one epoch; return the mean training loss as a float."""
    # 🟢 CODE HERE - 6 parts:
    #   1) model.train()
    #   2) two accumulators: total loss, number of samples seen
    #   3) for x, y in loader:  move both to device
    #   4) the five steps, in this order:
    #        optimizer.zero_grad() -> out = model(x) -> loss = criterion(out, y)
    #        -> loss.backward() -> optimizer.step()
    #   5) accumulate  loss.item() * x.size(0)  and  seen += x.size(0)
    #   6) return total / seen as a float   (averaged over samples, not over batches)
    model.train()
    total_loss = 0.0
    seen = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        seen += x.size(0)
    return total_loss / seen


# ⚪ RUN: one epoch on the 3000-image subset
model = make_model().to(DEVICE)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
t0 = time.time()
loss = train_one_epoch(model, train_loader, nn.CrossEntropyLoss(), opt, DEVICE)
print(f"epoch 1: mean loss = {loss:.4f}  ({time.time() - t0:.1f} s)")


train batches per epoch: 47 | test batches: 40


epoch 1: mean loss = 1.9694  (3.7 s)


## Question 2 · The full loop, and a scheduler

Write *fit(model, train_loader, test_loader, optimizer, epochs, device, scheduler=None)* that

1. calls *train_one_epoch* once per epoch,
2. then evaluates on *test_loader* **in eval mode** and records the accuracy,
3. calls *scheduler.step()* once per epoch, **after** the validation pass, when a scheduler is given,

and returns a dict *{"train_loss": [...], "test_acc": [...]}* with one entry per epoch
(*test_acc* between 0 and 1).


In [2]:
# 🟢 EDIT: write fit() using the train_one_epoch() from Q1.
# 🟡 READ: train_loader, test_loader, make_model() and DEVICE come from the Q1 cell above.
def fit(model, train_loader, test_loader, optimizer, epochs,
        device="cpu", scheduler=None):
    """Train for a few epochs; return {"train_loss": [...], "test_acc": [...]}.

    The model is left in eval mode; train_one_epoch() switches it back to train() at the
    start of every epoch, so calling fit() twice in a row is fine.
    """
    # 🟢 CODE HERE - one set-up line, then 5 parts per epoch:
    #   0) before the loop: hist = {"train_loss": [], "test_acc": []} and crit = nn.CrossEntropyLoss()
    #   1) loss = train_one_epoch(model, train_loader, crit, optimizer, device)   (reuse Q1)
    #   2) model.eval();  correct = total = 0
    #   3) with torch.no_grad():  loop over test_loader,
    #        pred = model(x).argmax(dim=1);  correct += (pred == y).sum().item();
    #        total += x.size(0)
    #   4) if scheduler is not None: scheduler.step()      once per epoch, after the evaluation
    #   5) append loss and correct / total to the two lists, then print one line per epoch
    #   return hist
    hist = {"train_loss": [], "test_acc": []}
    crit = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        loss = train_one_epoch(model, train_loader, crit, optimizer, device)
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += x.size(0)
        if scheduler is not None:
            scheduler.step()
        accuracy = correct / total
        hist["train_loss"].append(loss)
        hist["test_acc"].append(accuracy)
        print(f"epoch {epoch + 1}: loss = {loss:.4f}, test accuracy = {accuracy:.4f}")
    return hist


# ⚪ RUN: SGD and Adam, two epochs each
runs = {}
for name, opt_cls, lr in (("SGD", torch.optim.SGD, 0.05),
                          ("Adam", torch.optim.Adam, 1e-3)):
    torch.manual_seed(0)
    m = make_model().to(DEVICE)
    runs[name] = fit(m, train_loader, test_loader, opt_cls(m.parameters(), lr=lr),
                     epochs=2, device=DEVICE)
    print(f"{name}: loss {[round(v, 3) for v in runs[name]['train_loss']]}"
          f"  acc {[round(v, 3) for v in runs[name]['test_acc']]}")


epoch 1: loss = 1.9694, test accuracy = 0.3521


epoch 2: loss = 1.6734, test accuracy = 0.3772
SGD: loss [1.969, 1.673]  acc [0.352, 0.377]


epoch 1: loss = 2.1322, test accuracy = 0.3676


epoch 2: loss = 1.6277, test accuracy = 0.3688
Adam: loss [2.132, 1.628]  acc [0.368, 0.369]
